# BIST Swing / Scalp Trading Bot — Colab Ana Yürütücü

Bu notebook, projenin tüm `src/` modüllerini (veri yükleme, özellik mühendisliği,
XGBoost walk-forward modeli, risk yönetimi/backtest) tek bir yerden, hücre hücre
çalıştırmanızı sağlar.

**Nasıl kullanılır:**
1. Aşağıdaki "Ortam Kurulumu" hücrelerini sırayla çalıştırın (Drive bağlama,
   repo klonlama, bağımlılık kurulumu).
2. "Parametreler" hücresinde işlem yapmak istediğiniz BIST sembolünü ve zaman
   aralığını seçin.
3. Kalan hücreleri sırayla çalıştırarak veri çekme → özellik mühendisliği →
   walk-forward eğitim → backtest → canlı tahmin adımlarını izleyin.

> Not: Google Colab dışında (yerel Jupyter / VS Code) çalıştırıyorsanız Drive
> bağlama hücresi otomatik olarak atlanır; `src/config.py` ortamı kendisi algılar.


## 1. Ortam Kurulumu

### 1.1 Google Drive'ı bağla (yalnızca Colab'de)
Projeyi Google Drive üzerinde saklamak isterseniz bu hücreyi çalıştırın.
Yerel/GitHub Codespaces ortamında bu hücre otomatik olarak atlanır.


In [ ]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı.")
else:
    print("Colab ortamı algılanmadı; Drive bağlama adımı atlandı.")


### 1.2 Depoyu (repo) klonla / güncelle

`GITHUB_REPO_URL` değerini kendi reponuzun URL'si ile değiştirin. Repo zaten
klonlanmışsa (yerel geliştirme ortamı) bu hücre sadece dizine geçer ve `git pull`
çalıştırmayı dener.


In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/skumova/bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb.git"
REPO_DIR_NAME = "bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb"

if IN_COLAB:
    base_dir = Path("/content/drive/MyDrive") if Path("/content/drive/MyDrive").exists() else Path("/content")
else:
    base_dir = Path.cwd()

project_dir = base_dir / REPO_DIR_NAME

# Notebook zaten repo içinden çalıştırılıyorsa (örn. yerel geliştirme), mevcut
# dizini kullan; src/ klasörü burada bulunuyorsa yeniden klonlamaya gerek yok.
if (Path.cwd() / "src" / "config.py").exists():
    project_dir = Path.cwd()
elif project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "pull"], check=False)
else:
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(project_dir)], check=False)

os.chdir(project_dir)
print("Çalışma dizini:", Path.cwd())


### 1.3 Bağımlılıkları kur


In [ ]:
import sys

if IN_COLAB:
    !pip install -q -r requirements.txt
else:
    print("Colab dışı ortam: bağımlılıkların zaten kurulu olduğu varsayılıyor "
          "(gerekiyorsa `pip install -r requirements.txt` komutunu elle çalıştırın).")

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)


## 2. Modülleri İçe Aktar


In [ ]:
import logging
import numpy as np
import pandas as pd

from src import config
from src.data_loader import BistDataLoader
from src.features import build_feature_matrix, FEATURE_COLUMNS
from src.filters import build_filter_mask, FILTER_DIAGNOSTIC_COLUMNS
from src.model import WalkForwardEngine, predict_latest
from src.backtester import run_backtest, compute_financial_metrics

logging.getLogger("bist_bot").setLevel(logging.INFO)
print("Proje kökü:", config.PROJECT_ROOT)
print("Modüller başarıyla içe aktarıldı.")


## 3. Parametreler

İşlem yapmak istediğiniz BIST sembolünü, zaman dilimini ve walk-forward
ayarlarını burada belirleyin. TradingView kimlik bilgileriniz varsa
`TV_USERNAME` / `TV_PASSWORD` alanlarına girebilirsiniz (opsiyoneldir; boş
bırakılırsa tvdatafeed anonim modda dener, başarısız olursa otomatik olarak
yfinance kullanılır).


In [ ]:
SYMBOL = "GARAN"          # BIST sembolü (örn. GARAN, THYAO, ASELS ...)
EXCHANGE = "BIST"
INTERVAL = "15m"           # 1m, 5m, 15m, 30m, 1h, 4h, 1d
LOOKBACK_BARS = 3000

TV_USERNAME = None         # opsiyonel TradingView kullanıcı adı
TV_PASSWORD = None         # opsiyonel TradingView şifresi

# Walk-forward / model ayarları (hızlı test için küçük tutulmuştur;
# gerçek kullanımda src/config.py'deki varsayılanları kullanabilirsiniz)
TRAIN_BARS = 800
STEP_BARS = 150
MIN_TEST_BARS = 50
RETRAIN_EVERY = 2
OPTUNA_TRIALS = 15
USE_OPTUNA = True

# Ek filtreler ve gelişmiş risk yönetimi (kurumsal seviye) - src/config.py
USE_FILTERS = True                 # ADX/hacim/volatilite-rejimi/trend/seans filtrelerini uygula
POSITION_SIZE_MODE = config.POSITION_SIZE_MODE       # "fixed" veya "vol_target"
RISK_PER_TRADE_PCT = config.RISK_PER_TRADE_PCT
EXIT_PROBABILITY_THRESHOLD = config.EXIT_PROBABILITY_THRESHOLD
MAX_CONSECUTIVE_LOSSES = config.MAX_CONSECUTIVE_LOSSES
COOLDOWN_BARS_AFTER_LOSSES = config.COOLDOWN_BARS_AFTER_LOSSES
MAX_DAILY_LOSS_PCT = config.MAX_DAILY_LOSS_PCT

DEMO_MODE_ON_FAILURE = True  # veri kaynağına ulaşılamazsa sentetik demo verisiyle devam et


## 4. Veri Yükleme (Önbellek Destekli)

İlk çalıştırmada `LOOKBACK_BARS` kadar geçmiş veri indirilir ve hem bellekte
hem diskte (parquet) önbelleğe alınır. Bu hücreyi tekrar çalıştırdığınızda
tüm geçmiş yeniden inmez; sadece en güncel bar(lar) çekilip mevcut
DataFrame'e eklenir.


In [ ]:
def _generate_demo_ohlcv(n_bars: int, interval: str) -> pd.DataFrame:
    """Veri kaynağına ulaşılamadığında pipeline'ı göstermek için sentetik OHLCV üretir."""
    freq_map = {"1m": "1min", "5m": "5min", "15m": "15min", "30m": "30min", "1h": "1h", "4h": "4h", "1d": "1d"}
    rng = np.random.default_rng(42)
    idx = pd.date_range("2023-01-01", periods=n_bars, freq=freq_map.get(interval, "15min"))
    returns = rng.normal(0, 0.004, n_bars)
    trend = np.sin(np.arange(n_bars) / 40) * 0.0025
    close = 100 * np.exp(np.cumsum(returns + trend))
    high = close * (1 + rng.uniform(0, 0.003, n_bars))
    low = close * (1 - rng.uniform(0, 0.003, n_bars))
    open_ = close * (1 + rng.uniform(-0.001, 0.001, n_bars))
    volume = rng.uniform(1000, 5000, n_bars)
    demo_df = pd.DataFrame(
        {"open": open_, "high": high, "low": low, "close": close, "volume": volume}, index=idx
    )
    demo_df.index.name = "datetime"
    return demo_df


loader = BistDataLoader(exchange=EXCHANGE, interval=INTERVAL, tv_username=TV_USERNAME, tv_password=TV_PASSWORD)

try:
    ohlcv_df = loader.get_history(SYMBOL, n_bars=LOOKBACK_BARS)
    print(f"[{SYMBOL}] gerçek veri kaynağından {len(ohlcv_df)} bar yüklendi.")
except Exception as exc:
    if not DEMO_MODE_ON_FAILURE:
        raise
    print(f"Veri kaynağına ulaşılamadı ({exc}). Sentetik DEMO veri kullanılıyor.")
    ohlcv_df = _generate_demo_ohlcv(LOOKBACK_BARS, INTERVAL)

ohlcv_df.tail()


## 5. Özellik Mühendisliği (Teknik İndikatörler + GARCH)


In [ ]:
feature_df = build_feature_matrix(ohlcv_df)
print("Özellik matrisi boyutu:", feature_df.shape)
feature_df[FEATURE_COLUMNS].tail()


## 5b. Ek Filtreler (ADX, Hacim, Volatilite Rejimi, Trend Hizası, İşlem Seansı)

Bu filtreler, XGBoost olasılık eşiği geçilse bile piyasa koşulları uygun
değilse (yatay/gürültülü piyasa, düşük hacim, aşırı sakin/aşırı oynak
volatilite rejimi, karşı-trend, seans dışı) girişi engeller. `tradable`
kolonu backtester tarafından ek bir kapı olarak kullanılır. Herhangi bir
filtreyi gevşetmek/sıkılaştırmak isterseniz `src/config.py` içindeki ilgili
sabitleri değiştirin.


In [ ]:
if USE_FILTERS:
    feature_df = build_filter_mask(feature_df)
    print("Filtre sonrası işleme uygun (tradable) bar oranı: "
          f"{feature_df['tradable'].mean():.1%}")
    feature_df[FILTER_DIAGNOSTIC_COLUMNS + ['tradable']].tail()
else:
    print("USE_FILTERS=False: ek filtreler devre dışı, tüm barlar işleme uygun kabul edilecek.")


## 6. Walk-Forward Eğitim, Otomatik Optimizasyon ve Backtest

`WalkForwardEngine`, belirlenen `TRAIN_BARS` penceresiyle modeli eğitir,
`STEP_BARS` kadar ileriye tahmin yapar ve bu adımı tüm veri boyunca tekrarlar.
Her fold için precision/recall/f1 ve (backtester üzerinden) Sharpe/Drawdown/
Win-Rate hesaplanıp `logs/walk_forward_metrics.csv` dosyasına loglanır.


In [ ]:
engine = WalkForwardEngine(
    train_bars=TRAIN_BARS,
    step_bars=STEP_BARS,
    min_test_bars=MIN_TEST_BARS,
    retrain_every=RETRAIN_EVERY,
    optuna_trials=OPTUNA_TRIALS,
    use_optuna=USE_OPTUNA,
    financial_metrics_fn=compute_financial_metrics,
)

oos_predictions = engine.run(feature_df)
print("Walk-forward özeti:", engine.summary())
oos_predictions[["proba_up", "signal", "label", "fold_id"]].tail()


In [ ]:
metrics_log = pd.read_csv(config.LOG_DIR / "walk_forward_metrics.csv")
metrics_log


## 7. Tüm Out-of-Sample Dönem Üzerinde Genel Backtest

Walk-forward'ın ürettiği tüm out-of-sample tahminlerini birleştirip uçtan uca
tek bir backtest (ATR tabanlı stop-loss / trailing-stop / take-profit,
filtreler ve gelişmiş risk yönetimiyle) çalıştırır.


In [ ]:
full_backtest = run_backtest(
    oos_predictions,
    position_size_mode=POSITION_SIZE_MODE,
    risk_per_trade_pct=RISK_PER_TRADE_PCT,
    exit_probability_threshold=EXIT_PROBABILITY_THRESHOLD,
    max_consecutive_losses=MAX_CONSECUTIVE_LOSSES,
    cooldown_bars_after_losses=COOLDOWN_BARS_AFTER_LOSSES,
    max_daily_loss_pct=MAX_DAILY_LOSS_PCT,
)
print("Genel backtest metrikleri (filtreler + gelişmiş risk yönetimi ile):")
for k, v in full_backtest["metrics"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

full_backtest["trades"].tail()


### 7b. Karşılaştırma: Filtresiz/Sabit-Boyutlu (eski) vs Filtreli/Risk-Yönetimli (yeni)

Aynı walk-forward tahminleri üzerinde, eski (filtre yok, sabit %100
pozisyon, devre kesici yok) ve yeni (ADX/hacim/volatilite/trend/seans
filtreleri + risk-bazlı boyutlandırma + devre kesici) yapılandırmaları
karşılaştırır. Gerçek BIST-100 verisiyle yapılan testlerde yeni yapılandırma
maksimum drawdown'u belirgin şekilde azaltmıştır.


In [ ]:
baseline_backtest = run_backtest(
    oos_predictions.drop(columns=["tradable"], errors="ignore"),
    position_size_mode="fixed",
    position_size_pct=1.0,
    exit_probability_threshold=None,
    max_consecutive_losses=None,
    max_daily_loss_pct=None,
)

comparison = pd.DataFrame({
    "eski (filtresiz/sabit)": baseline_backtest["metrics"],
    "yeni (filtreli/risk-yönetimli)": full_backtest["metrics"],
}).T
comparison


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
baseline_backtest["equity_curve"].plot(ax=ax, label="eski (filtresiz/sabit)", alpha=0.7)
full_backtest["equity_curve"].plot(ax=ax, label="yeni (filtreli/risk-yönetimli)", alpha=0.9)
ax.set_title(f"{SYMBOL} — Walk-Forward OOS Equity Curve Karşılaştırması")
ax.set_ylabel("Sermaye")
ax.set_xlabel("Zaman")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Modeli Kaydet


In [ ]:
model_path = engine.save_model()
print("Eğitilmiş model kaydedildi:", model_path)


## 9. Canlı Döngü Simülasyonu (Tek Adım)

Gerçek kullanımda bu hücre periyodik olarak (örn. her yeni bar kapanışında)
çalıştırılır: `loader.get_latest_bar()` sadece en güncel barı çeker ve mevcut
DataFrame'e ekler (tüm geçmiş tekrar inmez), özellikler yeniden hesaplanır ve
model üzerinden güncel yukarı yön olasılığı üretilir.


In [ ]:
try:
    latest_bar = loader.get_latest_bar(SYMBOL)
    live_ohlcv = loader._memory[SYMBOL.upper()]
except Exception as exc:
    print(f"Canlı bar alınamadı ({exc}); mevcut (demo) veriyle devam ediliyor.")
    live_ohlcv = ohlcv_df

live_features = build_feature_matrix(live_ohlcv)
if USE_FILTERS:
    live_features = build_filter_mask(live_features)
latest_row = live_features.iloc[-1]

proba_up = predict_latest(engine.model, latest_row, engine.feature_columns)
is_tradable = bool(latest_row["tradable"]) if USE_FILTERS else True
signal = "AL (BUY)" if (proba_up >= config.ENTRY_PROBABILITY_THRESHOLD and is_tradable) else "BEKLE (HOLD)"

print(f"Son bar zamanı     : {latest_row.name}")
print(f"Son kapanış        : {latest_row['close']:.4f}")
print(f"Yukarı yön olasılığı: {proba_up:.2%}")
if USE_FILTERS:
    print(f"Filtre durumu (tradable): {is_tradable}")
    if not is_tradable:
        print("  -> Model AL diyor olsa bile piyasa koşulları (ADX/hacim/volatilite/trend/seans) uygun değil.")
print(f"Sinyal (eşik={config.ENTRY_PROBABILITY_THRESHOLD:.0%})   : {signal}")
if signal.startswith("AL"):
    atr_val = latest_row[f"atr_{config.ATR_WINDOW}"]
    print(f"  Önerilen Stop-Loss   : {latest_row['close'] - config.ATR_STOP_MULTIPLIER * atr_val:.4f}")
    print(f"  Önerilen Take-Profit : {latest_row['close'] + config.ATR_TAKE_PROFIT_MULTIPLIER * atr_val:.4f}")


---
### Notlar
- Bu notebook eğitim/araştırma amaçlıdır; gerçek para ile canlı emir
  gönderme (broker/API entegrasyonu) içermez.
- Üretim kullanımında Hücre 9'u bir zamanlayıcı (örn. `cron`, Colab'de bir
  `while True` + `time.sleep`) içine alarak periyodik çalıştırabilirsiniz.
- Walk-forward periyotları, ATR çarpanları, olasılık eşiği, ek filtreler
  (ADX/hacim/volatilite rejimi/trend/seans) ve devre kesici (ardışık kayıp
  cooldown'u, günlük zarar limiti) parametrelerinin tümü `src/config.py`
  üzerinden merkezi olarak yönetilir.
